# Reproducing Fedele, Tzovara et al. (2020) from dandiset 000576

Paper. Fedele T, Tzovara A, Steiger B, Hilfiker P, Grunwald T, Stieglitz L, Jokeit H, Sarnthein J
(2020). The relation between neuronal firing, local field potentials and hemodynamic activity in the
human amygdala in response to aversive dynamic visual stimuli. *NeuroImage* 213, 116705.
[doi:10.1016/j.neuroimage.2020.116705](https://doi.org/10.1016/j.neuroimage.2020.116705). The claims
file was written from the version of record (main text, legends, Tables 1 and 2, Methods); the
supplement was not supplied and carries no finding. The data descriptor of the same recordings
(Fedele et al. 2021, *Sci Data*, doi:10.1038/s41597-020-00790-x) was read for the layout of the data.

Dandiset. [DANDI:000576](https://dandiarchive.org/dandiset/000576/0.231010.1811), version
0.231010.1811. The paper (2020) predates the dandiset (2023) and cites no version of it. This is the
only published version, so it is pinned; the draft lists the same 27 assets with the same total size.

What the dandiset holds. For each of the 9 patients one session of the video task: the bipolar
iEEG of each implanted amygdala (14 channels in all) at 2 kHz as 17 epochs of 26 s, the trial table,
and for 5 patients the spike times of 42 sorted units. Each session is deposited twice, as
`_ecephys.nwb` and `_ecephys+image.nwb` (the second adds a link to the video); the data in each pair
are identical (checked below), and each is counted once. The BOLD time courses (clinical fMRI) and the
continuous microwire signals (needed for the microwire high gamma power, muHGP) are not in it.

| Claim | Tested here |
|---|---|
| fig1b-delta-decrease, fig1c-hgp-increase-first-2s, other-bands-no-effect | tested |
| table2-mi-values, fig1d-delta-hgp-mi-correlation, nsoz-delta-hgp-mi-inverse | tested |
| table2-delta-reduced-12-of-14, table2-hgp-mi-positive-nsoz, fig1e-hgp-not-delta-distinguishes-soz | tested |
| hgp-soz-sensitivity-specificity, fig1f-hgp-more-specific-than-delta | tested |
| neuron-count, fig3a-firing-increase-aversive, fig3a-firing-gradual-sustained | tested |
| fig3b-firing-hgp-no-condition-difference, fig3c-ssi-frv-correlation, fig4b-regression-firing-hgp | tested |
| fig3d-firing-correlations-modalities | tested in part (HGP only; muHGP and BOLD are not in the dandiset) |
| fig3b-firing-muhgp-higher-aversive | blocked: the continuous microwire signals are not in the dandiset |
| fig2-participant4-bold-correlations, table2-bold-correlations-group, table2-bold-correlation-values, bold-hgp-uncorrelated-low-mi, fig4c-regression-firing-bold, fig4d-hgp-bold-covariation | blocked: the BOLD time courses are not public |

Method details the claims file leaves open are taken from the paper and cited where they are used,
here and in the docstring of the helper module `amygdala.py`.

In [ ]:
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import yaml
from scipy import stats

import amygdala as A

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore")

started = time.time()
all_assets = A.assets()
ecephys = sorted([a for a in all_assets if a["path"].endswith("_ecephys.nwb")], key=lambda a: a["path"])
with ThreadPoolExecutor(9) as ex:
    sessions = list(ex.map(A.load_session, ecephys))
print(f"loaded {len(sessions)} sessions from dandiset {A.DANDISET_ID} version {A.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

claims_file = Path("../../claims/000576/10.1016-j.neuroimage.2020.116705.yaml")
claims = {c["id"]: c for c in yaml.safe_load(claims_file.read_text())["claims"]}

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
kinds = pd.Series([("video" if a["path"].endswith(".avi") else
                    "ecephys+image.nwb" if a["path"].endswith("+image.nwb") else "ecephys.nwb")
                   for a in all_assets]).value_counts()
print("Assets in the version:", len(all_assets))
print(kinds.to_string())


def same_data(asset):
    """Compare the iEEG, spikes and trials of an `_ecephys+image.nwb` file with its `_ecephys.nwb` twin."""
    twin = next(a for a in ecephys if a["path"] == asset["path"].replace("_ecephys+image.nwb", "_ecephys.nwb"))
    h, g = A.open_asset(twin["asset_id"]), A.open_asset(asset["asset_id"])
    st = lambda f: f["units/spike_times"][:] if "units" in f else np.array([])
    return {"file": asset["path"].split("/")[-1],
            "ieeg_identical": np.array_equal(h["processing/ecephys/LFP/ecephys.ieeg/data"][:],
                                             g["acquisition/ecephys.ieeg/data"][:]),
            "spikes_identical": np.array_equal(st(h), st(g)),
            "trials_identical": np.array_equal(h["intervals/trials/start_time"][:], g["intervals/trials/start_time"][:])}


with ThreadPoolExecutor(9) as ex:
    twins = pd.DataFrame(ex.map(same_data, [a for a in all_assets if a["path"].endswith("+image.nwb")]))
print("\nEach `_ecephys+image.nwb` file against its `_ecephys.nwb` twin:")
print(twins.to_string(index=False))

The nine `_ecephys+image.nwb` files repeat the iEEG, spikes and trials of the nine `_ecephys.nwb`
files exactly; each recording is counted once, from the `_ecephys.nwb` file.

In [ ]:
paper_table1 = {1: (31, "m"), 2: (48, "m"), 3: (19, "f"), 4: (22, "m"), 5: (34, "m"), 6: (23, "m"),
                7: (39, "m"), 8: (27, "f"), 9: (22, "m")}   # Table 1 of the paper: age, gender
rows = []
for s in sessions:
    for c, lab in enumerate(s["labels"]):
        n = A.table2_number(s["subject"], lab)
        rows.append({"subject": s["subject"], "age_file": s["age"], "age_paper": paper_table1[s["subject"]][0],
                     "sex_file": s["sex"], "sex_paper": paper_table1[s["subject"]][1], "channel": lab,
                     "table2_amygdala": n, "soz_file": s["soz"][c], "soz_table2": n in A.TABLE2_SOZ,
                     "location": s["locations"][c], "filtering": s["filtering"][c]})
chan = pd.DataFrame(rows)
print(chan.to_string(index=False))
print("\nChannels:", len(chan), "; SOZ flags agree with Table 2:", bool((chan.soz_file == chan.soz_table2).all()),
      f"({int(chan.soz_file.sum())} SOZ, {int((~chan.soz_file).sum())} nSOZ)")
print("Ages that differ from the paper's Table 1:",
      [r for r in chan.drop_duplicates("subject")[["subject", "age_file", "age_paper"]].to_dict("records") if r["age_file"] != f"P{r['age_paper']}Y"])
print("Channels whose location names the hippocampus or is unknown:",
      chan[~chan.location.str.contains("Amygdala")][["subject", "channel", "location"]].to_dict("records"))

The 14 bipolar channels are the 14 amygdalae of Table 2 (7 nSOZ, 7 SOZ), and the files' SOZ flags
agree with the stars of Table 2. Two ages differ from the paper's Table 1 and from the data
descriptor's (subject 2: 39 in the file, 48 in the paper; subject 8: 24 against 27). The `location`
of five channels does not name the amygdala: the right channels of subjects 1, 5 and 9 are labelled
"Right Hippocampus, rostral hippocampus", and both of subject 6 "unknown", while the labels (`AR`,
`AL`), the paper and the data descriptor place every one in the amygdala. The analysis follows the
paper and uses all 14.

In [ ]:
rows = []
for s in sessions:
    x = s["lfp"]
    d = np.abs(np.diff(x, axis=0))
    bounds = np.arange(1, s["n_epochs"]) * int(A.EPOCH_S * A.FS) - 1
    rows.append({"subject": s["subject"], "samples": x.shape[0], "channels": x.shape[1],
                 "stated_unit": s["lfp_unit"], "t_first": s["t"][0], "t_last": s["t"][-1],
                 "dt_unique_ms": np.unique(np.round(np.diff(s["t"]) * 1e3, 6)).tolist(),
                 "sd_value": round(float(x.std()), 1), "max_abs_value": round(float(np.abs(x).max()), 1),
                 "median_step": round(float(np.median(d)), 2), "p99_step": round(float(np.percentile(d, 99)), 2),
                 "median_step_at_epoch_bounds": round(float(np.median(d[bounds])), 2)})
print(pd.DataFrame(rows).to_string(index=False))

Every file holds 884000 samples per channel at 0.5 ms (2 kHz), 17 epochs of 26 s. The data are
stated in volts, but the values have a standard deviation of tens and reach hundreds to 1546: they
are microvolts (the data descriptor's figures are in microvolts). The unit does not enter any
result (powers are z-scored, correlations are ranks). The step between consecutive samples at the
16 epoch boundaries is a median 12 to 36 against 1.4 to 1.9 within epochs, above the 99th percentile:
the epochs are cut out of the recording and laid end to end, not one continuous trace, although
the timestamps run without a gap. Each epoch is therefore filtered on its own (helper).

In [ ]:
rows = []
for s in sessions:
    starts, stops = s["trial_start"], s["trial_stop"]
    rows.append({"subject": s["subject"], "trials": s["n_epochs"], "aversive": int(s["aversive"].sum()),
                 "neutral": int((~s["aversive"]).sum()), "order": "".join("A" if a else "N" for a in s["aversive"]),
                 "trial1": f"{starts[0]:.1f}-{stops[0]:.1f}", "trial2": f"{starts[1]:.1f}-{stops[1]:.1f}",
                 "start_minus_26k": sorted(set(np.round(starts - np.arange(len(starts)) * A.EPOCH_S, 3))),
                 "durations": sorted(set(np.round(stops - starts, 3))),
                 "ts_index_start_2": s["trial_ts"][1][0]})
print(pd.DataFrame(rows).to_string(index=False))

Every session has 17 trials alternating NEUTRAL and AVERSIVE, 9 and 8, as Methods 4.4 give (eight
AVERSIVE and nine NEUTRAL blocks). Trials 2 to 17 start 2 s into their 26 s epoch and last 24 s: the
trial table marks the block and leaves out the 2 s baseline clip before it (Methods 4.4, data
descriptor Fig. 1a). Trial 1 starts at 0 and lasts 26 s, including its baseline; this is taken as an
inconsistency of the table, and block onset is taken at 2 s in every epoch. That the block starts
there is consistent with the high gamma and firing responses shown below, which change after 2 s and
not before; it is not proven by them.

In [ ]:
U = A.units(sessions)
urows = []
for u in U:
    sp = u["spikes"]
    counts = np.histogram(sp, np.arange(18) * A.EPOCH_S)[0]
    urows.append({"subject": u["subject"], "unit": u["unit"], "channel": u["label"], "amygdala": u["amygdala"],
                  "soz": u["amygdala"] in A.TABLE2_SOZ, "spikes": len(sp),
                  "rate_442s": round(len(sp) / 442, 2), "rate_observed": round(len(sp) / (u["obs"].sum() * 26), 2),
                  "epochs_without_spikes": np.where(counts == 0)[0].tolist(),
                  "sorted": bool(np.all(np.diff(sp) >= 0)), "first": round(sp.min(), 2), "last": round(sp.max(), 2)})
units_df = pd.DataFrame(urows)
print(units_df.to_string(index=False))
print("\nUnits per subject:", units_df.groupby("subject").size().to_dict())
print("Units per amygdala (Table 2 number):", units_df.groupby("amygdala").size().to_dict())
print("obs_intervals of every unit:", sorted({(len(s_u["obs"]), float(s_u["obs"][0, 0]), float(s_u["obs"][-1, 1]))
                                              for s in sessions for s_u in s["units"]}))

42 units in 8 amygdalae of 5 patients (1 L and R, 2 L and R, 3 L, 4 L, 7 L and R). Three of these
amygdalae (1 R, 2 R, 7 R, 17 units) are SOZ amygdalae in Table 2, while Discussion 3.4 says there
were no microwire recordings from SOZ amygdalae; the neuron claims use all units, as the paper's
counts (41 neurons) do. Spike times are sorted and lie within
0 to 442 s. Every unit's `obs_intervals` claims all 17 epochs, but the 16 units of subject 7 have no
spike after epoch 11 (the last at 286 s), and 4 units of subject 2 have none in epoch 1, at rates of
2 to 5 Hz elsewhere. These epochs are taken as not recorded for those units (helper,
`observed_epochs`); only the first or last epochs are affected, so each unit's observed epochs are
contiguous.

In [ ]:
# Do the spikes and the iEEG share a clock? Spike-triggered average of the bipolar channel of the
# unit's own amygdala, against the same with spike times shifted by a random 1 to 5 s.
rng_check = np.random.default_rng(0)
lags = np.arange(-200, 201)            # samples, +-100 ms
sta_rows = []
lfp_of = {(s["subject"], lab): s["lfp"][:, c] for s in sessions for c, lab in enumerate(s["labels"])}
for u in U:
    x = lfp_of[(u["subject"], u["label"])]
    x = (x - x.mean()) / x.std()
    idx = np.round(u["spikes"] * A.FS).astype(int)
    idx = idx[(idx > 400) & (idx < len(x) - 400)]
    sta = x[idx[:, None] + lags].mean(0)
    sh = idx + rng_check.integers(2000, 10000)
    sh = sh[sh < len(x) - 400]
    sta_sh = x[sh[:, None] + lags].mean(0)
    sta_rows.append({"subject": u["subject"], "unit": u["unit"], "peak_abs_sta_within_5ms": np.abs(sta[190:211]).max(),
                     "peak_abs_sta_shifted_within_5ms": np.abs(sta_sh[190:211]).max()})
sta_df = pd.DataFrame(sta_rows)
print(sta_df.groupby("subject")[["peak_abs_sta_within_5ms", "peak_abs_sta_shifted_within_5ms"]].median().round(3).to_string())
print("Units whose STA peak within +-5 ms exceeds the shifted one:",
      int((sta_df.peak_abs_sta_within_5ms > sta_df.peak_abs_sta_shifted_within_5ms).sum()), "of", len(sta_df))
print("Wilcoxon, true against shifted:", stats.wilcoxon(sta_df.peak_abs_sta_within_5ms, sta_df.peak_abs_sta_shifted_within_5ms))

The spike-triggered average of the macro bipolar iEEG within 5 ms of the spike is larger than at
shifted spike times in 24 of 42 units (Wilcoxon p = 0.042), with medians by subject from 0.02 to 0.14
SD against 0.02 to 0.06. The lock is weak; the microwires protrude a few millimetres beyond the
bipolar contacts, so a weak lock is not evidence of an offset. The shared clock of spikes, iEEG and
trial table is consistent with the data (the firing and high gamma responses change at the block
onset of the trial table, in the claim sections below) but not shown.

## iEEG power: envelopes and the group test

Envelopes per band (Methods 4.8) for each of the 14 amygdalae, as the helper computes them. The
three readings of "z-scored all data within each trial" that the claims file lists decide the
outcomes: `trial` (over the whole 26 s epoch), `baseline` (mean and SD of the 2 s baseline clip of
the epoch) and `amygdala` (over all epochs of the amygdala). Two variants are shown beside them and
do not enter any count of readings: `trialsd` (the baseline-corrected envelope divided by its SD
within the trial; under `trial` the baseline correction of Methods 4.8 has no effect, as a constant
subtracted before a z-score) and `baselinepool` (the epoch's baseline mean, with the SD of the
baseline clips pooled over all epochs of the amygdala). The second is shown because the SD of
`baseline` rests on 20 samples of a 2 s clip smoothed over 1 s, roughly two independent values, so
`baseline` z-values are noisy; `baselinepool` shows which `baseline` results survive a stable SD.

The group test is run over the 24 s block (0 to 24 s from onset, the time base of Fig. 1, "locked to
stimulus onset"); the baseline clip is the same in both conditions and follows a block of the other
condition. This window is a choice stated here.

In [ ]:
HGP_VARIANTS = {"hgp_60_150": (60.0, 150.0), "hgp_60_250": (60.0, 250.0)}   # variants of the open upper edge
t_env = A.times()
block = t_env >= 0
tb = t_env[block]
ENV = {}
for s in sessions:
    for c, lab in enumerate(s["labels"]):
        n = A.table2_number(s["subject"], lab)
        ep = A.epochs(s["lfp"][:, c])
        for band in A.BANDS:
            ENV[(n, band)] = A.envelope(ep, band)
        for name, edges in HGP_VARIANTS.items():
            ENV[(n, name)] = A.envelope(ep, edges)
aversive_of = {s["subject"]: s["aversive"] for s in sessions}
NSOZ = [n for n in range(1, 15) if n not in A.TABLE2_SOZ]
SOZ = sorted(A.TABLE2_SOZ)
ALL = list(range(1, 15))
ZS = ["trial", "baseline", "amygdala"]          # the claims file's readings: these decide outcomes
ZV = ["trialsd", "baselinepool"]                # variants, shown only
ZALL = ZS + ZV
SETS = {"nSOZ": NSOZ, "all14": ALL}


def stack(band, z, amygdalae, base="1s"):
    Y, X, G, L = [], [], [], []
    for n in amygdalae:
        subj = A.TABLE2[n - 1][0]
        y = A.normalise(ENV[(n, band)], z, base)[:, block]
        Y.append(y); X.append(aversive_of[subj].astype(int)); G += [subj] * len(y); L += [n] * len(y)
    return np.vstack(Y), np.concatenate(X), np.array(G), np.array(L)


print("envelopes:", len(ENV), "(14 amygdalae x 6 bands and 2 HGP variants), each", ENV[(1, "delta")].shape, "epochs x samples at",
      A.STEP_S, "s; block samples", block.sum())
# Check of the fast mixed-model t against statsmodels MixedLM at a few time points.
y, x, g, l = stack("delta", "trialsd", ALL)
t_fast, df_fast = A.lmm_t(y, x, g)
cmp = []
for i in [10, 60, 120, 200]:
    d = pd.DataFrame({"power": y[:, i], "condition": x, "patient": g})
    fit = smf.mixedlm("power ~ condition", d, groups=d["patient"]).fit(reml=True)
    cmp.append({"time_s": round(tb[i], 2), "t_fast": round(t_fast[i], 3), "t_mixedlm": round(fit.tvalues["condition"], 3),
                "beta_fast": round(float(np.polyfit(x, y[:, i], 1)[0]), 3), "beta_mixedlm": round(fit.params["condition"], 3)})
print(pd.DataFrame(cmp).to_string(index=False))
# How many independent values the SD of the `baseline` reading rests on: the 2 s clip holds 20 samples
# of an envelope smoothed over 1 s. Lag-1 to lag-10 autocorrelation of the delta and HGP envelopes in the clip:
for band in ["delta", "hgp"]:
    e = np.vstack([ENV[(n, band)][:, ~block] for n in ALL])
    e = e - e.mean(1, keepdims=True)
    ac = [np.mean((e[:, :-k] * e[:, k:]).sum(1) / (e ** 2).sum(1)) for k in (1, 5, 10)]
    print(f"{band}: baseline-clip autocorrelation at 0.1, 0.5, 1 s lag {np.round(ac, 2).tolist()}; "
          f"ratio of the largest to the smallest baseline SD over trials, median over amygdalae "
          f"{np.median([np.ptp(np.log(ENV[(n, band)][:, ~block].std(1))) for n in ALL]):.2f} (natural log)")

The condition effect and its t from the helper's within-patient formula match statsmodels' MixedLM
with a random intercept per patient at the time points shown (the small differences in t come from
MixedLM's variance estimate when the between-patient variance is near zero). The envelope in the 2 s
baseline clip decorrelates within about 0.5 s (autocorrelation 0.03 at 0.5 s), so the SD of the
`baseline` reading rests on a few independent values, and it varies across the trials of one
amygdala by a factor of about 6.5 for delta and 4.5 for HGP (median over amygdalae of the ratio of
the largest to the smallest); `baselinepool` replaces it with the SD pooled over the amygdala's
baseline clips.

In [ ]:
def run_band(band, rng, zs=None):
    """All readings of the group test for one band: z-scoring x amygdala set x cluster statistic, for
    the mixed model and the unpaired t-test variant."""
    rows = []
    for z in (zs or ZALL):
        for set_name, amys in SETS.items():
            y, x, g, l = stack(band, z, amys)
            diff = y[x == 1].mean() - y[x == 0].mean()
            for model in ("lmm", "ttest"):
                pg = g if model == "lmm" else np.zeros(len(g), int)
                for stat in ("mass", "size"):
                    _, df, res = A.cluster_test(y, x, g, rng, model=model, stat=stat, perm_groups=pg,
                                                linked=l if model == "lmm" else None)
                    sig = [r for r in res if r["p"] < 0.05]
                    rows.append({"band": band, "z": z, "reading": "claims" if z in ZS else "variant",
                                 "set": set_name, "model": model, "stat": stat,
                                 "block_mean_A_minus_N": round(diff, 3),
                                 "sig_clusters": "; ".join(f"{'A>N' if r['sign'] > 0 else 'A<N'} {tb[r['start']] - 0.05:.1f} to {tb[r['stop'] - 1] + 0.05:.1f} s p={r['p']:.3f}" for r in sig) or "none",
                                 "n_sig_neg": sum(r["sign"] < 0 for r in sig), "n_sig_pos": sum(r["sign"] > 0 for r in sig),
                                 "s_neg": round(sum((r["stop"] - r["start"]) * A.STEP_S for r in sig if r["sign"] < 0), 1),
                                 "s_pos": round(sum((r["stop"] - r["start"]) * A.STEP_S for r in sig if r["sign"] > 0), 1),
                                 "first_pos_start": min([tb[r["start"]] - 0.05 for r in sig if r["sign"] > 0], default=np.nan),
                                 "smallest_p": round(min([r["p"] for r in res], default=1.0), 3),
                                 "clusters": res})
    return pd.DataFrame(rows)


SHOW = ["band", "z", "reading", "set", "model", "stat", "block_mean_A_minus_N", "n_sig_neg", "n_sig_pos", "s_neg", "s_pos", "smallest_p", "sig_clusters"]

## fig1b-delta-decrease

> In amygdalae outside the SOZ, delta power (1-4 Hz) is lower during AVERSIVE than during NEUTRAL
> sequences, and the reduction lasts through the AVERSIVE sequence.

Criterion: under the mixed model, at least one cluster with AVERSIVE below NEUTRAL at permutation
p < 0.05 and the AVERSIVE block mean below the NEUTRAL one, under every reading; blocked as
methods_underspecified when the readings disagree; partially reproduced when the clusters cover
less than half of the 24 s block.

In [ ]:
G = {}
G["delta"] = run_band("delta", np.random.default_rng(101))
d = G["delta"]
print(d[SHOW].to_string(index=False))
main = d[d.model == "lmm"].copy()
main["holds"] = (main.n_sig_neg > 0) & (main.block_mean_A_minus_N < 0)
mc = main[main.reading == "claims"]
print("\nMixed-model readings of the claims file where the claim holds:", int(mc.holds.sum()), "of", len(mc))
print(main.groupby(["z", "set"]).holds.all().unstack().to_string())
print("\nSeconds of the block covered by significant negative clusters (mixed model, cluster mass):")
print(main[main.stat == "mass"].pivot(index="z", columns="set", values="s_neg").to_string())
print("Abstract: 'lasting up to 18 s'; half the block is 12 s.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
yrng = []
for ax, (set_name, amys) in zip(axes, SETS.items()):
    for z, col in zip(ZALL, ["k", "C0", "C1", "C2", "C3"]):
        y, x, g, l = stack("delta", z, amys)
        curve = y[x == 1].mean(0) - y[x == 0].mean(0)
        ax.plot(tb, curve, color=col, label=z, ls="-" if z in ZS else ":")
        yrng.append((z, set_name, round(float(curve.min()), 2), round(float(curve.max()), 2)))
    ax.axhline(0, color="grey", lw=0.5)
    ax.set_title(f"delta, {set_name}: AVERSIVE minus NEUTRAL")
    ax.set_xlabel("time from block onset (s)")
axes[0].set_ylabel("z (each reading's units)")
axes[0].legend(fontsize=8)
plt.tight_layout()
print("Range of each plotted curve (z, set, min, max):", yrng)

**Result: blocked (methods_underspecified).** The readings of the claims file disagree, as the
criterion foresees: the claim holds in 6 of the 12 mixed-model readings. With the z-score over the
whole trial (`trial`) no reading finds a delta cluster, in either amygdala set (smallest cluster p
0.88 and 1.0). With `baseline` z-scoring AVERSIVE is below NEUTRAL in the 7 nSOZ amygdalae
(significant clusters covering 14.3 s of the block) and in all 14 (15.6 s). With `amygdala`
z-scoring the 14 amygdalae give 17.1 s and the 7 nSOZ amygdalae none (smallest p 0.072). The block
mean of AVERSIVE minus NEUTRAL is negative under every reading (-0.03 to -1.39). Where clusters are
found they cover 14 to 17 s of the 24 s block, against the abstract's "up to 18 s". The variants
agree with `baseline`: `baselinepool` 13.3 s (nSOZ) and 16.8 s (all 14), `trialsd` 7.0 s and 16.1 s.
The unpaired t-test variant gives the same pattern.

## fig1c-hgp-increase-first-2s

> High gamma power (above 60 Hz) is higher during AVERSIVE than during NEUTRAL sequences in the
> first 2 s of the sequence.

Criterion: as fig1b, for HGP; reproduced when a cluster with AVERSIVE above NEUTRAL at p < 0.05 lies
within, or begins within, the first 2 s after onset, under every reading.

In [ ]:
G["hgp"] = run_band("hgp", np.random.default_rng(102))
d = G["hgp"]
print(d[SHOW + ["first_pos_start"]].to_string(index=False))
lm = d[(d.model == "lmm") & (d.reading == "claims")]
tt_ = d[(d.model == "ttest") & (d.reading == "claims")]
print("\nMixed-model readings of the claims file with a positive cluster (p < 0.05) beginning in the first 2 s:",
      int(((lm.n_sig_pos > 0) & (lm.first_pos_start < 2)).sum()), "of", len(lm))
print("Unpaired t-test readings with one:", int(((tt_.n_sig_pos > 0) & (tt_.first_pos_start < 2)).sum()), "of", len(tt_))
for set_name, amys in SETS.items():
    for z in ZALL:
        y, x, g, l = stack("hgp", z, amys)
        tt, df = A.lmm_t(y, x, g)
        crit = stats.t.ppf(0.975, df)
        print(f"{set_name:6s} {z:12s} largest mixed-model t in 0-2 s: {tt[tb < 2].max():.2f} at {tb[tb < 2][tt[tb < 2].argmax()]:.2f} s "
              f"(threshold {crit:.2f}); points above threshold in 0-2 s: {int((tt[tb < 2] > crit).sum())}; in 2-24 s: {int((tt[tb >= 2] > crit).sum())}")

In [ ]:
# Variants of the open upper edge of the HGP band (the claims file's reading is a high-pass at 60 Hz):
# 60-150 Hz and 60-250 Hz band-pass. Shown, not used for the outcome.
for name in HGP_VARIANTS:
    G[name] = run_band(name, np.random.default_rng(110 + list(HGP_VARIANTS).index(name)))
    v = G[name][G[name].model == "lmm"]
    print(f"{name}: mixed-model readings with a positive cluster starting in 0-2 s: "
          f"{int(((v.n_sig_pos > 0) & (v.first_pos_start < 2)).sum())} of {len(v)}; any significant cluster: "
          f"{int(((v.n_sig_pos + v.n_sig_neg) > 0).sum())}; smallest cluster p {v.smallest_p.min():.3f}")
    print(v[v.sig_clusters != "none"][["z", "set", "stat", "sig_clusters"]].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, (set_name, amys) in zip(axes, SETS.items()):
    for z, col in zip(ZALL, ["k", "C0", "C1", "C2", "C3"]):
        y, x, g, l = stack("hgp", z, amys)
        tt, df = A.lmm_t(y, x, g)
        ax.plot(tb, tt, color=col, label=z, ls="-" if z in ZS else ":")
    ax.axhline(stats.t.ppf(0.975, df), color="grey", ls="--", lw=0.8)
    ax.axhline(-stats.t.ppf(0.975, df), color="grey", ls="--", lw=0.8)
    ax.axvline(2, color="r", lw=0.5)
    ax.set_title(f"HGP, {set_name}: mixed-model t, AVERSIVE vs NEUTRAL")
    ax.set_xlabel("time from block onset (s)")
axes[0].legend(fontsize=8)
plt.tight_layout()
print("All plotted t values are inside the axes (autoscaled).")

**Result: not reproduced.** None of the 12 mixed-model readings and none of the 12 unpaired t-test
readings of the claims file finds a significant HGP cluster, in the first 2 s or later, and neither
do the two z-scoring variants. The largest mixed-model t in the first 2 s is 3.01, at 0.95 s after
onset (`baseline`, nSOZ), above the cluster-forming threshold at 13 points, and that cluster's
permutation p is 0.062 (smallest of the readings 0.056). With the band limited to 60-150 or
60-250 Hz (variants) the only significant HGP clusters are AVERSIVE above NEUTRAL from about 6 to 8 s
(`baseline`, nSOZ, p 0.020 and 0.042) and, with `baselinepool`, 3.1 to 4.7 s, not in the first 2 s.

### Per-amygdala time courses and the other baselines of Methods 4.8

Each amygdala's AVERSIVE minus NEUTRAL difference in 2 s bins of the block, for delta and HGP under
the three z-scorings of the claims file, to compare with the group curves of Fig. 1B and 1C. Then
the two other baselines Methods 4.8 say were tested ("taking as baseline the mean of all epochs, or
the mean of the previous epoch. Results remained quantitatively similar"), which the claims file
lists as variants. Both subtract one constant per trial, which a z-score over the trial (`trial`) or
over the baseline clip (`baseline`) removes, so they can change only `amygdala` and the variants.
All are run, mixed model with cluster mass.

In [ ]:
edges2 = np.arange(0, 25, 2)
for band in ["delta", "hgp"]:
    for z in ZS:
        rows = []
        for n in ALL:
            yv = A.normalise(ENV[(n, band)], z)[:, block]
            av = aversive_of[A.TABLE2[n - 1][0]]
            dif = yv[av].mean(0) - yv[~av].mean(0)
            rows.append({"amygdala": n, "soz": n in A.TABLE2_SOZ,
                         **{f"{a}-{a + 2}": round(float(dif[(tb >= a) & (tb < a + 2)].mean()), 2) for a in edges2[:-1]}})
        tab = pd.DataFrame(rows)
        print(f"\n{band}, z-scoring {z}: AVERSIVE minus NEUTRAL per amygdala in 2 s bins (s from onset)")
        print(tab.to_string(index=False))
        print("  amygdalae with a negative 0-24 s mean:", int((tab.iloc[:, 2:].mean(1) < 0).sum()), "of 14; nSOZ with a positive 0-2 s bin:",
              int((tab[~tab.soz]["0-2"] > 0).sum()), "of 7; SOZ:", int((tab[tab.soz]["0-2"] > 0).sum()), "of 7")

rng_b = np.random.default_rng(120)
rows = []
for band in ["delta", "hgp"]:
    for base in ["1s", "allepochs", "prevepoch"]:
        for z in ZALL:
            for set_name, amys in SETS.items():
                y, x, g, l = stack(band, z, amys, base)
                _, _, res = A.cluster_test(y, x, g, rng_b, model="lmm", stat="mass", perm_groups=g, linked=l)
                sig = [r for r in res if r["p"] < 0.05]
                rows.append({"band": band, "baseline": base, "z": z, "set": set_name,
                             "block_mean_A_minus_N": round(float(y[x == 1].mean() - y[x == 0].mean()), 3),
                             "smallest_p": round(min([r["p"] for r in res], default=1.0), 3),
                             "sig_clusters": "; ".join(f"{'A>N' if r['sign'] > 0 else 'A<N'} {tb[r['start']] - 0.05:.1f} to {tb[r['stop'] - 1] + 0.05:.1f} s p={r['p']:.3f}" for r in sig) or "none"})
rb = pd.DataFrame(rows)
print("\nBaselines of Methods 4.8 (mixed model, cluster mass):")
print(rb.to_string(index=False))

The per-amygdala tables show no amygdala whose HGP rises in the first 2 s under every z-scoring:
under `baseline` 4 of the 7 nSOZ amygdalae have a positive 0 to 2 s bin, under `amygdala` 5 and
under `trial` 4, against 0 to 4 of the 7 SOZ amygdalae. The delta difference is negative over the
block in 10 to 12 of 14 amygdalae. The baselines of Methods 4.8 leave `trial` and `baseline`
unchanged, as they must. Under `amygdala` the mean of all epochs gives delta clusters of 1.6 s
(nSOZ) and 4.2 s (all 14) and HGP clusters from 3.2 s; the previous epoch gives delta clusters over
most of the block in both sets and an HGP cluster in the nSOZ amygdalae from 0.6 to 5.5 s
(p = 0.002). With the `trialsd` variant two more HGP clusters begin in the first 2 s, both in the
nSOZ amygdalae: 0.6 to 1.9 s with the mean of all epochs (p = 0.028) and 0.5 to 8.4 s with the
previous epoch (p = 0.002). These three are the only readings here with an HGP cluster that begins
in the first 2 s, and all use a baseline other than the 1 s clip. As the blocks
alternate, the previous epoch is always a block of the other condition, so this baseline adds the
neighbouring block's condition difference to every trial; it is a variant of the claims file and
does not change the outcomes above, but the results are not "quantitatively similar" across these
baselines here.

## other-bands-no-effect

> No AVERSIVE against NEUTRAL difference is found in the theta, alpha, beta and gamma (30-60 Hz) bands.

A claim of absence, judged on the paper's test (mixed model, cluster permutation, p < 0.05), under
the three z-scorings of the claims file and both amygdala sets; the two z-scoring variants are shown.

In [ ]:
for band in ["theta", "alpha", "beta", "gamma"]:
    G[band] = run_band(band, np.random.default_rng(103 + list(A.BANDS).index(band)))
ob = pd.concat([G[b] for b in ["theta", "alpha", "beta", "gamma"]])
print(ob[ob.model == "lmm"][SHOW].to_string(index=False))
lm = ob[(ob.model == "lmm") & (ob.stat == "mass")]
print("\nMixed model, cluster mass: bands with any significant cluster, by reading")
print(lm.assign(any_sig=(lm.n_sig_neg + lm.n_sig_pos) > 0).pivot_table(index="z", columns="set", values="any_sig", aggfunc="sum").to_string())
lmall = ob[ob.model == "lmm"].assign(any_sig=lambda v: (v.n_sig_neg + v.n_sig_pos) > 0)
dl_ = G["delta"][G["delta"].model == "lmm"].set_index(["z", "set", "stat"])
by_reading = lmall.groupby(["z", "set", "stat"]).any_sig.any().to_frame("other_band_cluster")
by_reading["delta_effect"] = [(dl_.loc[k].n_sig_neg > 0) and (dl_.loc[k].block_mean_A_minus_N < 0) for k in by_reading.index]
print("\nPer mixed-model reading: does another band have a significant cluster, and does delta show the effect of Fig. 1B?")
print(by_reading.to_string())
cl = by_reading.loc[ZS]
print("Readings of the claims file with no cluster in any other band:", int((~cl.other_band_cluster).sum()), "of", len(cl),
      "; of those, readings with the delta effect:", int((~cl.other_band_cluster & cl.delta_effect).sum()))
print("Readings of the claims file with the delta effect:", int(cl.delta_effect.sum()), "; of those with another band clean:",
      int((cl.delta_effect & ~cl.other_band_cluster).sum()))

**Result: not reproduced (resting on an assumption).** The readings of the claims file disagree:
under `trial` z-scoring no band has a cluster, while under `baseline` the mixed model finds clusters
with AVERSIVE below NEUTRAL in alpha in the 7 nSOZ amygdalae (13.3 to 15.9 s, p 0.016) and in theta
and alpha in all 14 (p 0.002 to 0.028), and under `amygdala` all four bands have clusters in all 14
amygdalae (none in the 7 nSOZ). The outcome rests on one assumption: the paper's pipeline is one under
which its own delta result of Fig. 1B appears, since the paper reports both from the same analysis.
Every reading of the claims file that shows the delta effect (`baseline` in both sets, `amygdala` in
all 14; 6 of 12) also shows a cluster in another band, and the `trial` reading, the only one with no
other-band cluster, shows no delta effect either. Of the variants, `baselinepool` in the 7 nSOZ
amygdalae shows the delta effect and no other-band cluster (the alpha cluster of `baseline` does not
survive a pooled baseline SD); in all 14 it has theta and alpha clusters.

## Modulation indices (table2-mi-values and the claims built on them)

MI = AVERSIVE minus NEUTRAL time-averaged power over the window where the conditions differ
(Methods 4.11, Table 2 legend), per amygdala. Window readings: (a) the group clusters of the band
(Fig. 1B and 1C) under the same z-scoring, from the mixed model with cluster mass over the 7 nSOZ
amygdalae (`a_nSOZ`) or all 14 (`a_all14`); (b) clusters found per amygdala (unpaired t-test of its
8 AVERSIVE against 9 NEUTRAL trials, cluster mass, 500 permutations); and (c), a reading the
Results' text gives, fixed windows: "the temporal window of task-dependent modulation, which
occurred at stimulus onset in HGP and was sustained throughout the trial in DELTA" (Results 2.1,
paragraph 2) and "HGP enhancement in the first 2 s or sustained delta decrease" (Methods 4.11), read
as 0 to 2 s for HGP and 0 to 24 s for delta. (a) and (b) are the claims file's readings; (c) is taken
from the paper's text as step 1.6 of the protocol allows, and is labelled so in every table.

Where a reading gives no significant window (no group cluster under (a), no cluster in that
amygdala under (b)), the MI is left undefined: the paper defines no MI without a window, and Table 2
prints a non-zero MI for each of the 14 amygdalae. A claim is judged only on readings that define
every MI it needs; the number of amygdalae each reading defines is printed.

In [ ]:
rng_mi = np.random.default_rng(201)


def group_mask(band, z, set_name):
    g = G[band]
    r = g[(g.z == z) & (g.set == set_name) & (g.model == "lmm") & (g.stat == "mass")].iloc[0]
    m = np.zeros(block.sum(), bool)
    for c in r["clusters"]:
        if c["p"] < 0.05:
            m[c["start"]:c["stop"]] = True
    return m


per_amy_cache = {}


def per_amygdala_mask(n, band, z):
    key = (n, band, z)
    if key not in per_amy_cache:
        y = A.normalise(ENV[(n, band)], z)[:, block]
        x = aversive_of[A.TABLE2[n - 1][0]].astype(int)
        _, _, res = A.cluster_test(y, x, np.zeros(len(x), int), rng_mi, model="ttest", stat="mass")
        m = np.zeros(block.sum(), bool)
        for c in res:
            if c["p"] < 0.05:
                m[c["start"]:c["stop"]] = True
        per_amy_cache[key] = m
    return per_amy_cache[key]


def mi(n, band, z, window):
    y = A.normalise(ENV[(n, band)], z)[:, block]
    x = aversive_of[A.TABLE2[n - 1][0]]
    if window.startswith("a_"):
        m = group_mask(band, z, "nSOZ" if window == "a_nSOZ" else "all14")
    elif window == "b":
        m = per_amygdala_mask(n, band, z)
    else:
        m = tb < 2 if band == "hgp" else np.ones(block.sum(), bool)
    if not m.any():
        return np.nan
    return y[x][:, m].mean() - y[~x][:, m].mean()


WINDOWS = ["a_nSOZ", "a_all14", "b", "c"]
MI = pd.DataFrame([{"amygdala": n, "z": z, "window": w, "delta": mi(n, "delta", z, w), "hgp": mi(n, "hgp", z, w)}
                   for n in ALL for z in ZALL for w in WINDOWS])
exp = claims["table2-mi-values"]["expected"]
printed = pd.DataFrame({"amygdala": ALL, "delta_paper": [e["value"] for e in exp[0::2]],
                        "hgp_paper": [e["value"] for e in exp[1::2]],
                        "delta_tol": [e["tolerance"] for e in exp[0::2]], "hgp_tol": [e["tolerance"] for e in exp[1::2]]})
MI = MI.merge(printed, on="amygdala")
MI["soz"] = MI.amygdala.isin(A.TABLE2_SOZ)
MI["delta_ok"] = (MI.delta - MI.delta_paper).abs() <= MI.delta_tol
MI["hgp_ok"] = (MI.hgp - MI.hgp_paper).abs() <= MI.hgp_tol
MI["delta_sign_ok"] = np.sign(MI.delta) == np.sign(MI.delta_paper)
MI["hgp_sign_ok"] = np.sign(MI.hgp) == np.sign(MI.hgp_paper)
summary = MI.groupby(["z", "window"]).agg(within_tol=("delta_ok", "sum"), hgp_within=("hgp_ok", "sum"),
                                          delta_sign=("delta_sign_ok", "sum"), hgp_sign=("hgp_sign_ok", "sum"),
                                          delta_defined=("delta", lambda v: int(v.notna().sum())),
                                          hgp_defined=("hgp", lambda v: int(v.notna().sum())))
summary["values_within_tol_of_28"] = summary.within_tol + summary.hgp_within
summary["signs_agree_of_28"] = summary.delta_sign + summary.hgp_sign
summary["reading"] = ["claims" if z in ZS and w != "c" else "claims z, window from text" if z in ZS else "variant"
                      for z, w in summary.index]
print(summary[["reading", "values_within_tol_of_28", "signs_agree_of_28", "delta_defined", "hgp_defined"]].to_string())
print("\nPer amygdala under the fixed windows (c), every z-scoring, with Table 2:")
wide = MI[MI.window == "c"].pivot(index="amygdala", columns="z", values=["delta", "hgp"]).round(2)
wide["delta_paper"] = printed.set_index("amygdala").delta_paper
wide["hgp_paper"] = printed.set_index("amygdala").hgp_paper
print(wide.to_string())
print("\nPer amygdala under (a), group windows (NaN: no group cluster):")
print(MI[MI.window.str.startswith("a_")].pivot(index="amygdala", columns=["window", "z"], values="delta").round(2).to_string())
print("\nPer amygdala under (b), per-amygdala clusters (NaN: no cluster in that amygdala):")
print(MI[MI.window == "b"].pivot(index="amygdala", columns="z", values=["delta", "hgp"]).round(2).to_string())

**Result: not reproduced.** No reading gives all 28 values within tolerance: at best 4 of 28 among
the claims file's z-scorings (`amygdala`, window c; 6 with the `trialsd` variant). No reading gives
the 28 signs of Table 2: at best 23 of 28 (`amygdala`, window c). Window (a) defines no HGP MI, as
there is no HGP group cluster, and window (b) defines only 1 to 3 MIs per band.

### Does the dandiset's patient numbering match the paper's?

Two hints that it may not: the units are in subjects 1, 2, 3, 4 and 7 while Table 1 lists MUA for 1,
2, 3, 4, 5 R and 8 L, and subject 2's file gives age 39, the age Table 1 gives for patient 7 (both
are gliosis cases with AL, AR implanted and AR in the SOZ, so they could be exchanged without the
SOZ flags showing it). The MIs are compared with Table 2 under the file numbering and under two
re-mappings: subjects 2 and 7 swapped (amygdalae 3 and 4 against 10 and 11), and subject 7's
amygdalae taken as patients 8 L and 5 R (10 against 12, 11 against 7).

In [ ]:
MAPS = {"file numbering": {}, "subjects 2 and 7 swapped": {3: 10, 4: 11, 10: 3, 11: 4},
        "subject 7 as 8 L and 5 R": {10: 12, 12: 10, 11: 7, 7: 11}}
rows = []
for name, sw in MAPS.items():
    target = [sw.get(n, n) for n in ALL]
    pdl = printed.set_index("amygdala").loc[target]
    for z in ZALL:
        m = MI[(MI.z == z) & (MI.window == "c")].sort_values("amygdala")
        dl, hg = m.delta.values, m.hgp.values
        rows.append({"mapping": name, "z": z, "window": "c",
                     "delta_sign_agree": int((np.sign(dl) == np.sign(pdl.delta_paper.values)).sum()),
                     "hgp_sign_agree": int((np.sign(hg) == np.sign(pdl.hgp_paper.values)).sum()),
                     "within_tol_of_28": int((np.abs(dl - pdl.delta_paper.values) <= pdl.delta_tol.values).sum()
                                             + (np.abs(hg - pdl.hgp_paper.values) <= pdl.hgp_tol.values).sum()),
                     "spearman_delta_vs_table2": round(stats.spearmanr(dl, pdl.delta_paper.values).statistic, 2),
                     "spearman_hgp_vs_table2": round(stats.spearmanr(hg, pdl.hgp_paper.values).statistic, 2)})
print(pd.DataFrame(rows).to_string(index=False))

Neither re-mapping brings the MIs clearly closer to Table 2. Under window c and the claims file's
z-scorings the signs agree in 16, 19 and 23 of 28 under the file numbering, 20, 21 and 23 with
subjects 2 and 7 swapped, and 18, 19 and 23 with subject 7 as patients 8 L and 5 R; the values within
tolerance stay at 0 to 4 of 28. The analysis keeps the file numbering, which the SOZ flags, sexes,
pathologies and implanted sides of the files support (Table 1).

## Claims on the MI values

Each claim below is computed under every z-scoring and window. A row counts towards the outcome when
its z-scoring is one of the claims file's (`trial`, `baseline`, `amygdala`) and it defines every MI the
claim needs; rows of the variant z-scorings and rows with undefined MIs are shown and marked. Window
(a) defines no HGP MI (there is no HGP group cluster), and window (b) defines only the 1 to 3
amygdalae per band that have a cluster of their own, so most claims rest on window (c). The printed
Table 2 values are run through the same computations as a reference.

In [ ]:
soz_mask = np.array([n in A.TABLE2_SOZ for n in ALL])
paper_delta, paper_hgp = printed.delta_paper.values, printed.hgp_paper.values


def readings(need=("delta", "hgp"), amys=None):
    """(z, window, kind, delta MI, HGP MI, defined) for every reading; `defined` when every MI in
    `need` is defined for the amygdalae `amys` (boolean mask over the 14, default all)."""
    sel = np.ones(14, bool) if amys is None else amys
    for z in ZALL:
        for w in WINDOWS:
            m = MI[(MI.z == z) & (MI.window == w)].sort_values("amygdala")
            dl, hg = m.delta.values, m.hgp.values
            ok = all(np.isfinite({"delta": dl, "hgp": hg}[b][sel]).all() for b in need)
            kind = "variant" if z in ZV else "claims"
            yield z, w, kind, dl, hg, ok


def counted(df):
    """Rows that decide the outcome: claims-file z-scoring and every needed MI defined."""
    return df[(df.kind == "claims") & df.defined]

### fig1d-delta-hgp-mi-correlation

> Across the 14 amygdalae, the DELTA modulation index and the HGP modulation index are negatively
> correlated.

Criterion: Spearman r within 0.067 of -0.67 and p < 0.05 under every reading; partially reproduced
when r is negative with p < 0.05 but outside tolerance.

In [ ]:
rows = []
for z, w, kind, dl, hg, ok in readings():
    r = stats.spearmanr(dl, hg) if ok else (np.nan, np.nan)
    rows.append({"z": z, "window": w, "kind": kind, "defined": ok, "r": r[0], "p": r[1]})
rows.append({"z": "Table 2 printed", "window": "", "kind": "paper", "defined": True,
             **dict(zip(["r", "p"], stats.spearmanr(paper_delta, paper_hgp)))})
r1d = pd.DataFrame(rows)
r1d["within_tol"] = (r1d.r + 0.67).abs() <= 0.067
print(r1d.round(3).to_string(index=False))
c = counted(r1d)
print(f"Counted readings: {len(c)}; r from {c.r.min():.2f} to {c.r.max():.2f}; within tolerance with p < 0.05: "
      f"{int((c.within_tol & (c.p < 0.05)).sum())}; negative with p < 0.05: {int(((c.r < 0) & (c.p < 0.05)).sum())}")

**Result: not reproduced.** The three readings that define both MIs (window c under `trial`,
`baseline` and `amygdala`) give r = -0.30, -0.12 and -0.12 (p 0.30, 0.69, 0.68), negative but far
from -0.67 and not significant; the variants give -0.33 and -0.04. The printed Table 2 values give
r = -0.74 (p = 0.003).

This outcome rests on an assumption: window (c), fixed windows taken from the Results' text, is the only window that defines the MIs it needs, while the paper defines the MI over the interval where the two curves are significantly separated (Table 2 legend). The question is for the authors (methods_unavailable).

### nsoz-delta-hgp-mi-inverse

> Among the 7 amygdalae outside the SOZ, the stronger the HGP enhancement (HGP MI), the stronger the
> reduction of DELTA power (more negative DELTA MI).

Criterion: Spearman r between DELTA MI and HGP MI over the 7 nSOZ amygdalae negative under every
reading.

In [ ]:
rows = []
for z, w, kind, dl, hg, ok in readings(amys=~soz_mask):
    r = stats.spearmanr(dl[~soz_mask], hg[~soz_mask]) if ok else (np.nan, np.nan)
    rows.append({"z": z, "window": w, "kind": kind, "defined": ok, "r": r[0], "p": r[1]})
rows.append({"z": "Table 2 printed", "window": "", "kind": "paper", "defined": True,
             **dict(zip(["r", "p"], stats.spearmanr(paper_delta[~soz_mask], paper_hgp[~soz_mask])))})
rn = pd.DataFrame(rows)
print(rn.round(3).to_string(index=False))
c = counted(rn)
print("Counted readings with r < 0:", int((c.r < 0).sum()), "of", len(c))

**Result: not reproduced.** Over the 7 nSOZ amygdalae, window c gives r = -0.57 (`trial`), -0.14
(`baseline`) and +0.07 (`amygdala`): negative in 2 of the 3 readings that define the MIs, so not
under every reading. Windows (a) and (b) define no HGP MI for these amygdalae. The variants give
-0.07 and +0.21. The printed Table 2 values give r = -0.68 (p = 0.094).

This outcome rests on an assumption: window (c), fixed windows taken from the Results' text, is the only window that defines the MIs it needs, while the paper defines the MI over the interval where the two curves are significantly separated (Table 2 legend). The question is for the authors (methods_unavailable).

### table2-delta-reduced-12-of-14

> DELTA power is reduced (negative MI) in 12 of the 14 amygdalae, whether inside or outside the SOZ.

Criterion: 11 to 13 negative DELTA MI under every reading, with negative MI both inside and outside
the SOZ.

In [ ]:
rows = []
for z, w, kind, dl, hg, ok in readings(need=("delta",)):
    d_ = dl if ok else np.full(14, np.nan)
    rows.append({"z": z, "window": w, "kind": kind, "defined": ok, "delta_defined": int(np.isfinite(dl).sum()),
                 "negative": int((d_ < 0).sum()) if ok else np.nan,
                 "negative_nSOZ": int((d_[~soz_mask] < 0).sum()) if ok else np.nan,
                 "negative_SOZ": int((d_[soz_mask] < 0).sum()) if ok else np.nan,
                 "positive_amygdalae": [n for n, v in zip(ALL, dl) if v > 0]})
rows.append({"z": "Table 2 printed", "window": "", "kind": "paper", "defined": True, "delta_defined": 14,
             "negative": int((paper_delta < 0).sum()),
             "negative_nSOZ": int((paper_delta[~soz_mask] < 0).sum()), "negative_SOZ": int((paper_delta[soz_mask] < 0).sum()),
             "positive_amygdalae": [n for n, v in zip(ALL, paper_delta) if v > 0]})
r12 = pd.DataFrame(rows)
r12["holds"] = r12.negative.between(11, 13) & (r12.negative_nSOZ > 0) & (r12.negative_SOZ > 0)
print(r12.to_string(index=False))
c = counted(r12)
print("Counted readings where it holds:", int(c.holds.sum()), "of", len(c), "; failing:",
      c[~c.holds][["z", "window", "negative", "negative_nSOZ", "negative_SOZ"]].to_dict("records"))

**Result: partially reproduced.** In 5 of the 6 readings that define all 14 delta MIs the count is
11 or 12, with negative MIs both outside (6 of 7) and inside (5 or 6 of 7) the SOZ: `baseline` with
windows a (both sets) and c (11; positive in amygdalae 2, 10 and 11), and `amygdala` with windows
a (all 14) and c (12; positive in 2 and 10). Under `trial` with window c the count is 10 (positive
in 2, 9, 10 and 11), one below the range, so it does not hold under every reading. Table 2 has
amygdalae 2 and 4 positive. Window (b) defines 1 or 2 delta MIs. The variants give 10 to 13.

This outcome rests on an assumption: its only failing reading uses window (c), fixed windows taken
from the Results' text and not from the claims file. The readings of the claims file that define all
14 delta MIs (window a under `baseline` in both sets and under `amygdala` in all 14) give 11, 11
and 12, all in range. The question is for the authors (methods_unavailable).

### table2-hgp-mi-positive-nsoz

> All 7 amygdalae outside the SOZ have a positive HGP MI, while the response is absent in amygdalae
> within the SOZ (6 of 7 negative).

Criterion: both counts match (7 of 7, 6 of 7) and the difference has p < 0.05 under one of the
tests: (a) Wilcoxon rank-sum, (b) unpaired t-test, (c) Fisher's exact test of the counts of
positive MI (chi-square shown beside it).

In [ ]:
rows = []
for z, w, kind, dl, hg, ok in list(readings(need=("hgp",))) + [("Table 2 printed", "", "paper", paper_delta, paper_hgp, True)]:
    if not ok:
        rows.append({"z": z, "window": w, "kind": kind, "defined": ok, "hgp_defined": int(np.isfinite(hg).sum())})
        continue
    pos_n, neg_s = int((hg[~soz_mask] > 0).sum()), int((hg[soz_mask] < 0).sum())
    table = [[pos_n, 7 - pos_n], [int((hg[soz_mask] > 0).sum()), 7 - int((hg[soz_mask] > 0).sum())]]
    rows.append({"z": z, "window": w, "kind": kind, "defined": ok, "hgp_defined": 14, "nSOZ_positive": pos_n, "SOZ_negative": neg_s,
                 "p_ranksum": stats.ranksums(hg[~soz_mask], hg[soz_mask]).pvalue,
                 "p_ttest": stats.ttest_ind(hg[~soz_mask], hg[soz_mask]).pvalue,
                 "p_fisher": stats.fisher_exact(table).pvalue,
                 "p_chi2": stats.chi2_contingency(table, correction=False)[1] if min(map(sum, table)) > 0 and min(map(sum, zip(*table))) > 0 else np.nan})
rh = pd.DataFrame(rows)
rh["counts_match"] = (rh.nSOZ_positive == 7) & (rh.SOZ_negative == 6)
print(rh.round(4).to_string(index=False))
c = counted(rh)
print("Counted readings:", len(c), "; with both counts matching:", int(c.counts_match.sum()))

**Result: not reproduced.** The readings that define every HGP MI (window c) give 4, 4 and 5 of 7
nSOZ amygdalae with a positive HGP MI and 3, 5 and 7 of 7 SOZ amygdalae with a negative one
(`trial`, `baseline`, `amygdala`); none gives 7 and 6. Under `amygdala` Fisher's test of the counts
gives p = 0.021, but the nSOZ count is 5, not 7.

This outcome rests on an assumption: window (c), fixed windows taken from the Results' text, is the only window that defines the MIs it needs, while the paper defines the MI over the interval where the two curves are significantly separated (Table 2 legend). The question is for the authors (methods_unavailable).

### fig1e-hgp-not-delta-distinguishes-soz

> The HGP MI, but not the DELTA MI, distinguishes amygdalae within the SOZ from those outside it.

Criterion: under a test, (a) Wilcoxon rank-sum or (b) unpaired t-test, uncorrected, HGP MI differs
between nSOZ and SOZ with p < 0.05 and DELTA MI does not; not reproduced when HGP does not differ or
DELTA does under every test reading; blocked when the test readings disagree.

In [ ]:
rows = []
for z, w, kind, dl, hg, ok in list(readings()) + [("Table 2 printed", "", "paper", paper_delta, paper_hgp, True)]:
    for test, fn in [("ranksum", lambda a, b: stats.ranksums(a, b).pvalue), ("ttest", lambda a, b: stats.ttest_ind(a, b).pvalue)]:
        if not ok:
            rows.append({"z": z, "window": w, "kind": kind, "defined": ok, "test": test})
            continue
        ph, pdl = fn(hg[~soz_mask], hg[soz_mask]), fn(dl[~soz_mask], dl[soz_mask])
        rows.append({"z": z, "window": w, "kind": kind, "defined": ok, "test": test, "p_hgp": ph, "p_delta": pdl,
                     "hgp_nSOZ_minus_SOZ": np.mean(hg[~soz_mask]) - np.mean(hg[soz_mask]),
                     "delta_nSOZ_minus_SOZ": np.mean(dl[~soz_mask]) - np.mean(dl[soz_mask]),
                     "holds": ph < 0.05 and pdl >= 0.05})
re_ = pd.DataFrame(rows)
print(re_.round(4).to_string(index=False))
c = counted(re_)
print("Counted rows (readings x test) where it holds:", int(c.holds.fillna(False).sum()), "of", len(c),
      f"; smallest HGP p {c.p_hgp.min():.3f}")

**Result: not reproduced.** Under the readings that define both MIs (window c) and neither test does
the HGP MI differ between nSOZ and SOZ amygdalae with p < 0.05 (smallest p 0.084, `baseline`
t-test); the DELTA MI does not differ either (p 0.08 to 0.95). With the `baselinepool` variant the
rank-sum test gives HGP p = 0.035 and DELTA p = 0.48 (t-test HGP p = 0.052). The printed Table 2
values give HGP p = 0.004 and 0.002, and DELTA p = 0.035 and 0.013, so by these tests the printed
DELTA MI also differs.

This outcome rests on an assumption: window (c), fixed windows taken from the Results' text, is the only window that defines the MIs it needs, while the paper defines the MI over the interval where the two curves are significantly separated (Table 2 legend). The question is for the authors (methods_unavailable).

### hgp-soz-sensitivity-specificity

> HGP enhancement indicates amygdalar functional integrity (outside the SOZ) with sensitivity 7/7
> (100%), specificity 6/7 (86%) and accuracy 13/14 (93%).

Criterion: the effect is a positive HGP MI; counts must match; Clopper-Pearson 95% intervals.

In [ ]:
rows = []
for z, w, kind, dl, hg, ok in list(readings(need=("hgp",))) + [("Table 2 printed", "", "paper", paper_delta, paper_hgp, True)]:
    if not ok:
        rows.append({"z": z, "window": w, "kind": kind, "defined": ok})
        continue
    tp, fn_ = int((hg[~soz_mask] > 0).sum()), int((hg[~soz_mask] <= 0).sum())
    fp, tn = int((hg[soz_mask] > 0).sum()), int((hg[soz_mask] <= 0).sum())
    rows.append({"z": z, "window": w, "kind": kind, "defined": ok, "TP": tp, "FN": fn_, "FP": fp, "TN": tn,
                 "sensitivity": f"{tp}/7", "specificity": f"{tn}/7", "accuracy": f"{tp + tn}/14",
                 "CI_sens": tuple(round(100 * v) for v in A.clopper_pearson(tp, 7)),
                 "CI_spec": tuple(round(100 * v) for v in A.clopper_pearson(tn, 7)),
                 "CI_acc": tuple(round(100 * v) for v in A.clopper_pearson(tp + tn, 14))})
rs = pd.DataFrame(rows)
rs["counts_match"] = (rs.TP == 7) & (rs.TN == 6)
print(rs.to_string(index=False))
c = counted(rs)
print("Counted readings:", len(c), "; with the counts matching:", int(c.counts_match.sum()))
print("Paper: 7/7 CI [66-100], 6/7 CI [42-99], 13/14 CI [66-99]")
# Other binomial intervals, to see which one the printed bounds follow.
from statsmodels.stats.proportion import proportion_confint
for k, n_ in [(7, 7), (6, 7), (13, 14)]:
    cp1 = (stats.beta.ppf(0.05, k, n_ - k + 1) if k > 0 else 0.0, stats.beta.ppf(0.95, k + 1, n_ - k) if k < n_ else 1.0)
    print(f"{k}/{n_}: two-sided Clopper-Pearson {tuple(round(100 * v) for v in A.clopper_pearson(k, n_))}; "
          f"one-sided 95% bounds {tuple(round(100 * v) for v in cp1)}; Wilson {tuple(round(100 * v) for v in proportion_confint(k, n_, method='wilson'))}; "
          f"Jeffreys {tuple(round(100 * v) for v in proportion_confint(k, n_, method='jeffreys'))}")

**Result: not reproduced.** Window c gives sensitivity 4/7, 4/7 and 5/7, specificity 3/7, 5/7 and
7/7, accuracy 7/14, 9/14 and 12/14 (`trial`, `baseline`, `amygdala`); none gives 7/7, 6/7 and 13/14.
The two-sided Clopper-Pearson intervals of the paper's counts (last row) are 59-100, 42-100 and
66-100%, against the printed 66-100, 42-99 and 66-99%. No single interval method gives all three
printed intervals: the one-sided 95% bound gives 65 for 7/7, Wilson gives 65-100, 49-97 and 69-99.

This outcome rests on an assumption: window (c), fixed windows taken from the Results' text, is the only window that defines the MIs it needs, while the paper defines the MI over the interval where the two curves are significantly separated (Table 2 legend). The question is for the authors (methods_unavailable).

### fig1f-hgp-more-specific-than-delta

> The HGP response is more specific than the DELTA response for indicating amygdalar dysfunction
> within the SOZ, by a chi-square test that is not significant (p about 0.2).

Criterion: reading (a), over the 7 SOZ amygdalae, HGP effect absent (HGP MI not positive) against
DELTA effect absent (DELTA MI not negative), chi-square of the 2 x 2 table; reproduced when HGP
specificity exceeds DELTA specificity under (a); (b), the printed counts 5/5 against 1/5, is
reported; blocked if the readings disagree on which is more specific.

In [ ]:
rows = []
for z, w, kind, dl, hg, ok in list(readings(amys=soz_mask)) + [("Table 2 printed", "", "paper", paper_delta, paper_hgp, True)]:
    if not ok:
        rows.append({"z": z, "window": w, "kind": kind, "defined": ok})
        continue
    sh, sd = int((hg[soz_mask] <= 0).sum()), int((dl[soz_mask] >= 0).sum())
    table = np.array([[sh, 7 - sh], [sd, 7 - sd]])
    p = stats.chi2_contingency(table, correction=False)[1] if (table.sum(0) > 0).all() else np.nan
    rows.append({"z": z, "window": w, "kind": kind, "defined": ok, "HGP_specific": f"{sh}/7", "DELTA_specific": f"{sd}/7",
                 "which": "HGP" if sh > sd else "DELTA" if sd > sh else "tie", "p_chi2": p})
rf = pd.DataFrame(rows)
print(rf.round(3).to_string(index=False))
pb = stats.chi2_contingency(np.array([[5, 0], [1, 4]]), correction=False)[1]
pby = stats.chi2_contingency(np.array([[5, 0], [1, 4]]), correction=True)[1]
print(f"Reading (b), the printed counts 5/5 against 1/5: chi-square p = {pb:.4f} (with Yates' correction {pby:.4f}); paper p ~ 0.2")
c = counted(rf)
print("Counted readings:", len(c), "; more specific:", c.which.value_counts().to_dict())

**Result: blocked (methods_underspecified).** The readings disagree on which response is more
specific. With window c, the only window that defines the MIs of all 7 SOZ amygdalae, HGP is more
specific under `baseline` (5/7 against 2/7, chi-square p = 0.11) and `amygdala` (7/7 against 1/7,
p = 0.001), and the two are tied under `trial` (3/7 each). The variants also give HGP (6/7 against
1/7 and 3/7). Under reading (b), the printed counts, HGP is more specific with p = 0.0098 (0.053 with
Yates' correction), not about 0.2; no reading gives a p near 0.2.

This outcome rests on an assumption: window (c), fixed windows taken from the Results' text, is the only window that defines the MIs it needs, while the paper defines the MI over the interval where the two curves are significantly separated (Table 2 legend). The question is for the authors (methods_unavailable).

## Neurons

### neuron-count

> Spike sorting gave 50 putative neuron clusters from 9 amygdalae, of which 41 had a firing rate above
> 1 Hz and were analysed.

Criterion: mean rate above 1 Hz, over (a) all trials or (b) the whole recording, as the dataset allows;
41 within 4.1, 50 within 5, 9 amygdalae exactly.

In [ ]:
print("Units in the dandiset:", len(units_df))
print("Units with rate > 1 Hz over all 17 trials (442 s):", int((units_df.rate_442s > 1).sum()))
print("Units with rate > 1 Hz over the epochs each unit was recorded in:", int((units_df.rate_observed > 1).sum()))
print("Units at or below 1 Hz over 442 s:", units_df[units_df.rate_442s <= 1][["subject", "unit", "rate_442s", "rate_observed"]].to_dict("records"))
print("Amygdalae with units:", units_df.amygdala.nunique(), sorted(units_df.amygdala.unique()),
      "; patients with units:", units_df.subject.nunique(), sorted(units_df.subject.unique()))
print("Rates over observed epochs: mean %.2f, SD %.2f, min %.2f, max %.2f Hz (data descriptor: 1.66 +- 2.65 Hz)"
      % (units_df.rate_observed.mean(), units_df.rate_observed.std(), units_df.rate_observed.min(), units_df.rate_observed.max()))
print("Paper Table 1 lists MUA in 1 L/R, 2 L/R, 3 L, 4 L, 5 R, 8 L; units here are in subjects",
      units_df.groupby("subject").channel.unique().apply(list).to_dict())

**Result: partially reproduced.** The dandiset holds 42 units: 42 above 1 Hz over the epochs each
was recorded in and 40 over the whole 442 s, both within 4.1 of 41. The 50 putative clusters cannot
be tested, as only units above about 1 Hz were deposited (two at 0.88 Hz over the 442 s). The units come from 8 amygdalae of 5 patients,
not 9 amygdalae. They are not the amygdalae of Table 1: subject 7 (L and R) has units here and none
in Table 1, and subjects 5 (R) and 8 (L) have them in Table 1 and none here. Their mean rate is
4.65 +- 3.91 Hz (median 3.13), against the data descriptor's 1.66 +- 2.65 Hz; the table below gives
each unit's rate in the baseline clips and in each condition's blocks.

In [ ]:
# Per-unit rates by part of the epoch, and the population PSTH, to compare with the data descriptor's
# 1.66 +- 2.65 Hz (its Fig. 2c, "average firing rate for all neurons").
t100 = np.arange(260) * 0.1 + 0.05 - A.ONSET_S
rows = []
for u in U:
    r, o, av = u["rate100"], u["obs"], u["aversive"]
    rows.append({"subject": u["subject"], "unit": u["unit"], "amygdala": u["amygdala"],
                 "baseline_clip": r[o][:, t100 < 0].mean(), "aversive_block": r[o & av][:, t100 >= 0].mean(),
                 "neutral_block": r[o & ~av][:, t100 >= 0].mean(), "all_observed": r[o].mean(), "over_442s": r.mean()})
rates = pd.DataFrame(rows)
print(rates.round(2).to_string(index=False))
print("\nMean +- SD over units:", {k: f"{rates[k].mean():.2f} +- {rates[k].std(ddof=1):.2f}" for k in rates.columns[3:]})
print("Median over units:", rates[rates.columns[3:]].median().round(2).to_dict())
print("Units with rate above 1, 2 and 5 Hz over the observed epochs:", [int((rates.all_observed > k).sum()) for k in (1, 2, 5)])
fig, ax = plt.subplots(figsize=(7, 3))
for cond, col in [(True, "C3"), (False, "C0")]:
    psth = np.mean([A.smooth_epochs(u["rate100"], 10)[u["obs"] & (u["aversive"] == cond)].mean(0) for u in U], axis=0)
    ax.plot(t100, psth, color=col, label="AVERSIVE" if cond else "NEUTRAL")
ax.axvline(0, color="k", lw=0.5); ax.set_xlabel("time from block onset (s)"); ax.set_ylabel("mean rate over units (Hz)"); ax.legend(fontsize=8)
print("Population PSTH plotted (100 ms bins, 1 s smoothing), range within the autoscaled axes.")

### fig3a-firing-increase-aversive

> All 41 neurons fire more during AVERSIVE than during NEUTRAL sequences (FRV 0.26 plus or minus 0.15).

Criterion: FRV (AVERSIVE minus NEUTRAL mean rate) over (a) the 24 s blocks or (b) the 26 s trials;
reproduced when every neuron has FRV > 0 and the mean is within 0.15 of 0.26 under one reading;
partially reproduced when the mean is within tolerance and most neurons have FRV > 0.

In [ ]:
rows = []
for window in ["block", "trial"]:
    for only in [True, False]:
        f = np.array([A.frv(u, window, only) for u in U])
        rows.append({"window": window, "observed_epochs_only": only, "n": len(f), "mean_FRV_Hz": f.mean(), "sd": f.std(ddof=1),
                     "positive": int((f > 0).sum()), "wilcoxon_p": stats.wilcoxon(f).pvalue})
rfrv = pd.DataFrame(rows)
print(rfrv.round(4).to_string(index=False))
FRV = np.array([A.frv(u, "block") for u in U])
units_df["FRV"] = FRV
print("\nFRV by subject (blocks, observed epochs):")
print(units_df.groupby("subject").FRV.agg(["count", "mean", "min", "max", lambda v: int((v > 0).sum())]).round(3).to_string())
# Variant (not the claim's measure): FRV relative to the unit's mean rate, and in units of the SD of its 100 ms rate.
rel = np.array([A.frv(u, "block") / u["rate100"][u["obs"]].mean() for u in U])
zfrv = np.array([A.frv(u, "block") / u["rate100"][u["obs"]].std() for u in U])
print(f"\nVariant: FRV / mean rate {rel.mean():.3f} +- {rel.std(ddof=1):.3f} ({(rel > 0).sum()} positive); "
      f"FRV / SD of 100 ms rate {zfrv.mean():.3f} +- {zfrv.std(ddof=1):.3f}")

**Result: not reproduced.** Over the blocks and the observed epochs, 27 of 42 units have FRV > 0,
mean 0.033 +- 0.690 Hz (Wilcoxon against 0, p = 0.50); the other readings give 23 to 26 positive and
means of 0.007 to 0.032 Hz. The mean is outside 0.26 +- 0.15 and not all units fire more in AVERSIVE.
Relative to each unit's mean rate the FRV is 0.061 +- 0.231 (a variant, not the claim's measure).

### fig3a-firing-gradual-sustained

> The neurons' firing rose gradually and stayed enhanced throughout the AVERSIVE sequences, not as an
> immediate response at onset.

Criterion: population AVERSIVE minus NEUTRAL rate (100 ms bins, 1 s smoothing), (a) the difference
over 0 to 2 s smaller than over 2 to 24 s and positive in each of the twelve 2 s bins; (b) its first
peak later than 2 s and positive to 24 s; reproduced when both hold, not reproduced when both fail.

In [ ]:
tr = np.arange(260) * 0.1 + 0.05 - A.ONSET_S
diffs = []
for u in U:
    r = A.smooth_epochs(u["rate100"], 10)
    o, av = u["obs"], u["aversive"]
    diffs.append(r[o & av].mean(0) - r[o & ~av].mean(0))
diffs = np.array(diffs)
pop = diffs.mean(0)
pb = pop[tr >= 0]
tblk = tr[tr >= 0]
bins2 = [pb[(tblk >= k) & (tblk < k + 2)].mean() for k in range(0, 24, 2)]
early, late = pb[tblk < 2].mean(), pb[tblk >= 2].mean()
print("Population AVERSIVE - NEUTRAL (Hz) in 2 s bins:", np.round(bins2, 3).tolist())
print(f"(a) mean 0-2 s {early:.3f}, mean 2-24 s {late:.3f}; early < late: {early < late}; all twelve bins positive: {all(b > 0 for b in bins2)}")
from scipy.signal import find_peaks
pk = find_peaks(pb)[0]
first_peak = tblk[pk[0]] if len(pk) else np.nan
print(f"(b) first local peak at {first_peak:.2f} s; later than 2 s: {first_peak > 2}; positive at every point 2-24 s: {bool((pb[tblk >= 2] > 0).all())}; "
      f"fraction of 0-24 s positive {np.mean(pb > 0):.2f}")
print(f"Baseline clip (-2 to 0 s) mean difference {pop[tr < 0].mean():.3f} Hz")
print("Per subject, mean 2-24 s difference:", {s: round(float(diffs[[u['subject'] == s for u in U]][:, tr >= 2].mean()), 3) for s in sorted(set(u['subject'] for u in U))})
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(tr, pop, "k")
ax.fill_between(tr, pop - diffs.std(0, ddof=1) / np.sqrt(len(U)), pop + diffs.std(0, ddof=1) / np.sqrt(len(U)), color="grey", alpha=0.3)
ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="r", lw=0.5)
ax.set_xlabel("time from block onset (s)"); ax.set_ylabel("AVERSIVE - NEUTRAL (Hz)")
print(f"Plotted range {pop.min():.3f} to {pop.max():.3f} Hz (mean +- SEM band within the autoscaled axes)")

**Result: not reproduced.** The population AVERSIVE minus NEUTRAL rate is highest at onset: 0.310 Hz
over 0 to 2 s against 0.018 Hz over 2 to 24 s, its first peak at 0.15 s, and 4 of the twelve 2 s
bins negative (positive at 57% of the time points). Neither reading of "gradual and sustained" holds.
The difference is -0.078 Hz over the baseline clip, so the rise begins at the block onset of the
trial table, consistent with that onset.

### fig3b-firing-hgp-no-condition-difference

> Across neurons, the correlation of firing rate with macro-electrode HGP does not differ between the
> AVERSIVE and NEUTRAL conditions.

A claim of absence: Wilcoxon signed-rank test of the per-neuron Spearman r (AVERSIVE against NEUTRAL)
at p < 0.05. HGP is the Methods 4.8 envelope of the bipolar channel of the neuron's amygdala, without
per-trial normalisation (a Spearman r within one block is unchanged by any per-trial normalisation).
The claims file's readings: time bases (a) 1 s, (b) 1.5 s with 5 points, (c) 15 s; spans (i) the
condition's blocks laid end to end, (ii) per block and averaged. In (i) each block is smoothed (or
binned into 1.5 s bins) on its own before the blocks are laid end to end, so that no sample of a
condition's series carries the other condition or a baseline clip. A variant, `concat_session`,
smooths over the whole session first (the first revision's form), to show what that mixing does.

In [ ]:
HGP100 = {n: ENV[(n, "hgp")] for n in ALL}
rows = []
per_neuron = {}
for base in ["1s", "1.5s", "15s"]:
    for span in ["concat", "block", "concat_session"]:
        rr = np.array([A.condition_corr(u["rate100"], HGP100[u["amygdala"]], u["obs"], u["aversive"], base, span) for u in U])
        ok = ~np.isnan(rr).any(1)
        per_neuron[(base, span)] = rr
        w = stats.wilcoxon(rr[ok, 0], rr[ok, 1])
        rows.append({"base": base, "span": span, "kind": "variant" if span == "concat_session" else "claims", "n": int(ok.sum()),
                     "r_aversive": f"{np.nanmean(rr[:, 0]):.3f} +- {np.nanstd(rr[:, 0], ddof=1):.3f}",
                     "r_neutral": f"{np.nanmean(rr[:, 1]):.3f} +- {np.nanstd(rr[:, 1], ddof=1):.3f}",
                     "median_diff": np.median(rr[ok, 0] - rr[ok, 1]), "wilcoxon_p": w.pvalue})
r3b = pd.DataFrame(rows)
print(r3b.round(4).to_string(index=False))
c = r3b[r3b.kind == "claims"]
print("Readings of the claims file with a difference at p < 0.05:", int((c.wilcoxon_p < 0.05).sum()), "of", len(c))
print("Paper: 0.17 +- 0.28 AVERSIVE, 0.18 +- 0.18 NEUTRAL, Wilcoxon p > 0.05")
rr = per_neuron[("1s", "concat")]
print("\n1 s, blocks laid end to end, by subject, mean r AVERSIVE and NEUTRAL:")
print(pd.DataFrame({"subject": units_df.subject, "r_A": rr[:, 0], "r_N": rr[:, 1]}).groupby("subject").agg(["count", "mean"]).round(3).to_string())

**Result: partially reproduced.** The readings of the claims file disagree. Per block and averaged,
no time base finds a difference (p 0.72 to 0.83), and neither does the 1 s time base over the blocks
laid end to end (r = 0.085 +- 0.130 in AVERSIVE and 0.094 +- 0.124 in NEUTRAL, p = 0.54). Over the
blocks laid end to end on the 1.5 s and 15 s time bases, each block smoothed on its own, the test
finds NEUTRAL higher (p = 0.012 and 0.023, median differences -0.087 and -0.131). So the claimed
absence holds in 4 of the 6 readings. Smoothing over the whole session before selecting the blocks
(variant `concat_session`) gives p = 0.002 and 0.009 for those two, so the mixing of neighbouring
blocks strengthened the difference but does not explain it. The means are within the stated spreads
under every reading.

The outcome is partially reproduced, not blocked: the six readings are named by the claims file, so
no method detail is left to choose; the test was run under each, and the claimed absence holds under
some and fails under others, that is, it holds only in part. Unlike the criteria of
fig1b-delta-decrease and fig1f-hgp-more-specific-than-delta, this criterion does not say to block
when its readings disagree.

### fig3c-ssi-frv-correlation

> Across neurons, the spatial synchronization index (SSI) is strongly positively correlated with the
> firing rate variability (FRV).

SSI per neuron is the mean Spearman correlation of its firing rate with each neuron of the same
amygdala on a different microwire (Methods 4.11, "neighboring neurons recorded in the same amygdala
but on different microwires"). The units table is printed first, to see whether it records the
microwire. Where it does not, every other unit of the amygdala is used, an assumption (the Results
say "all neighboring neurons"), and a variant drops the pairs that are likely on one wire by a spike
train test: units sorted from one microwire cannot both be detected within about 1 ms of each other,
so a pair with no spikes within +-1 ms where the flanks (2 to 12 ms) predict at least 3 is taken to
share a wire. Time bases (a) 100 ms smoothed over 1 s, (b) 15 s, over the whole session (the epochs
both units were recorded in). FRV over the blocks.

In [ ]:
h_units = A.open_asset(next(a for a in ecephys if a["path"].startswith("sub-07"))["asset_id"])
print("Units table of subject 7, columns:", list(h_units["units"].attrs["colnames"]))
print("Electrode rows the units point to:", h_units["units/electrodes"][:].tolist(),
      "; electrode table labels:", [A._s(x) for x in h_units["general/extracellular_ephys/electrodes/label"][:]],
      "; electrode groups in the file:", [k for k in h_units["general/extracellular_ephys"].keys() if k != "electrodes"])

pairs = []
for i, u in enumerate(U):
    for j, v in enumerate(U):
        if j <= i or v["subject"] != u["subject"] or v["amygdala"] != u["amygdala"]:
            continue
        near, expected = A.coincidences(u["spikes"], v["spikes"])
        pairs.append({"i": i, "j": j, "subject": u["subject"], "amygdala": u["amygdala"], "near_1ms": near,
                      "expected": round(expected, 1), "likely_same_wire": near == 0 and expected >= 3,
                      "testable": expected >= 3})
pairs = pd.DataFrame(pairs)
print(f"\nPairs in one amygdala: {len(pairs)}; testable (>= 3 expected within +-1 ms): {int(pairs.testable.sum())}; "
      f"likely on one wire: {int(pairs.likely_same_wire.sum())}")
print(pairs.groupby("amygdala")[["testable", "likely_same_wire"]].sum().astype(int).to_string())
print("Ratio near/expected over testable pairs, quartiles:",
      np.round(np.percentile((pairs.near_1ms / pairs.expected)[pairs.testable], [25, 50, 75]), 2).tolist())
same_wire = {(r.i, r.j) for r in pairs.itertuples() if r.likely_same_wire}

SSI = {}
for base in ["1s", "15s"]:
    for drop in (False, True):
        vals = []
        for i, u in enumerate(U):
            rs_ = []
            for j, v in enumerate(U):
                if j == i or v["subject"] != u["subject"] or v["amygdala"] != u["amygdala"]:
                    continue
                if drop and (min(i, j), max(i, j)) in same_wire:
                    continue
                o = u["obs"] & v["obs"]
                a, _, _ = A.series(u["rate100"], o, base)
                b, _, _ = A.series(v["rate100"], o, base)
                rs_.append(A.spearman(a, b))
            vals.append(np.mean(rs_) if rs_ else np.nan)
        SSI[base + ("_xwire" if drop else "")] = np.array(vals)
units_df["SSI_1s"], units_df["SSI_15s"] = SSI["1s"], SSI["15s"]
print("\nUnits with an SSI (another unit in the same amygdala):", int(np.isfinite(SSI["1s"]).sum()), "of", len(U),
      "; after dropping likely same-wire pairs:", int(np.isfinite(SSI["1s_xwire"]).sum()))
for base in SSI:
    ok = np.isfinite(SSI[base])
    r = stats.spearmanr(SSI[base][ok], FRV[ok])
    print(f"base {base}{' (variant)' if 'xwire' in base else ''}: SSI mean {np.nanmean(SSI[base]):.3f} "
          f"(range {np.nanmin(SSI[base]):.3f} to {np.nanmax(SSI[base]):.3f}); Spearman SSI vs FRV r = {r.statistic:.3f}, p = {r.pvalue:.2g}, n = {ok.sum()}")
    if "xwire" in base:
        continue
    for s in sorted(units_df.subject.unique()):
        m = ok & (units_df.subject.values == s)
        if m.sum() > 2:
            rr = stats.spearmanr(SSI[base][m], FRV[m])
            print(f"   subject {s}: n = {m.sum()}, r = {rr.statistic:.3f}, p = {rr.pvalue:.3f}")
print("Paper: r = 0.78, p < 1e-7")

**Result: not reproduced (resting on an assumption).** The units table records no microwire: each
unit points only to the bipolar macro channel of its amygdala, and the file holds one electrode
group. SSI therefore uses every other unit of the amygdala (the assumption). The spike train test
finds 4 of 116 testable pairs likely on one wire; the other pairs fire within 1 ms of each other about
as often as the flanks predict (median ratio 1.19). SSI and FRV are positively correlated, but
weakly: r = 0.26 (p = 0.11) on the 1 s time base and r = 0.40 (p = 0.009) on the 15 s one, over the 41
units with another unit in their amygdala, against r = 0.78, p < 1e-7. Dropping the 4 pairs gives
0.24 (p = 0.13) and 0.35 (p = 0.023). The criterion needs p < 0.05 under every reading for a partial
result.

### fig3d-firing-correlations-modalities (HGP part)

> Across neurons, the ongoing firing rate correlates positively with muHGP (0.27), HGP (0.19) and
> BOLD (0.23), and the correlation is lowest with HGP.

Per neuron, Spearman r of firing rate with the HGP envelope of its amygdala over the session
(observed epochs, baseline clips included), (a) 15 s smoothing (Fig. 3D legend), (b) 1.5 s bins
smoothed over 5 points (Methods 4.10). muHGP and BOLD are not in the dandiset, so the means for them
and the two Wilcoxon tests cannot be computed.

In [ ]:
RH = {}
for base in ["15s", "1.5s", "1s"]:
    vals = []
    for u in U:
        a, _, _ = A.series(u["rate100"], u["obs"], base)
        b, _, _ = A.series(HGP100[u["amygdala"]], u["obs"], base)
        vals.append(A.spearman(a, b))
    RH[base] = np.array(vals)
    print(f"base {base}: firing-HGP r = {np.nanmean(RH[base]):.3f} +- {np.nanstd(RH[base], ddof=1):.3f} "
          f"(n = {np.isfinite(RH[base]).sum()}, {int((RH[base] > 0).sum())} positive); Wilcoxon against 0 p = {stats.wilcoxon(RH[base][np.isfinite(RH[base])]).pvalue:.3g}")
print("Paper: HGP 0.19 +- 0.17 (tolerance 0.17)")
units_df["r_hgp_15s"] = RH["15s"]
print(units_df.groupby("subject").r_hgp_15s.agg(["count", "mean", "min", "max"]).round(3).to_string())

**Result: partially reproduced.** The firing-HGP correlation is positive on average, 0.136 +- 0.241 on
the 15 s time base and 0.139 +- 0.195 on the 1.5 s one (positive in 29 and 32 of 42 units, Wilcoxon
against 0 p = 6e-4 and 2e-5), within the stated 0.19 +- 0.17. The muHGP and BOLD means, and the tests
that HGP is lowest, cannot be computed from the dandiset.

### fig4b-regression-firing-hgp

> The per-neuron correlation of firing rate with HGP is described by both FRV and SSI in a multiple
> linear regression.

Dependent variable the per-neuron firing-HGP r (fig3d readings), independent FRV and SSI (fig3c
readings, same time base where both exist). r read as (a) the multiple correlation R, (b) Spearman r
of fitted against observed. Units with an SSI only.

In [ ]:
rows = []
for rbase in ["15s", "1.5s"]:
    for sbase in ["1s", "15s", "1s_xwire", "15s_xwire"]:
        ok = np.isfinite(SSI[sbase]) & np.isfinite(RH[rbase])
        X = sm.add_constant(np.column_stack([FRV[ok], SSI[sbase][ok]]))
        fit = sm.OLS(RH[rbase][ok], X).fit()
        rows.append({"r_base": rbase, "ssi_base": sbase, "kind": "variant" if "xwire" in sbase else "claims", "n": int(ok.sum()),
                     "R": np.sqrt(fit.rsquared), "model_p": f"{fit.f_pvalue:.2e}",
                     "spearman_fit_obs": stats.spearmanr(fit.fittedvalues, RH[rbase][ok]).statistic,
                     "b_FRV": fit.params[1], "p_FRV": fit.pvalues[1], "b_SSI": fit.params[2], "p_SSI": fit.pvalues[2]})
r4b = pd.DataFrame(rows)
print(r4b.round(4).to_string(index=False))
print("FRV and SSI across these units: Spearman r =", {b: round(stats.spearmanr(FRV[np.isfinite(SSI[b])], SSI[b][np.isfinite(SSI[b])]).statistic, 3) for b in SSI})
c = r4b[r4b.kind == "claims"]
print("Readings of the claims file with both coefficients p < 0.05:", int(((c.p_FRV < 0.05) & (c.p_SSI < 0.05)).sum()), "of", len(c))
print("Paper: r = 0.59, p < 1e-4, both FRV and SSI")

**Result: not reproduced (resting on an assumption).** The regression describes the firing-HGP
correlation: R = 0.56 to 0.68 over the four time-base readings, with model p from 6e-6 to 7e-4; on the
15 s time bases R = 0.635 (within 0.059 of 0.59) with model p = 5.6e-5. But SSI carries it alone
(p <= 2e-4 in every reading) and FRV does not (p 0.07 to 0.65, coefficient negative in three
readings), so "described by both FRV and SSI", the claim and the criterion's condition, fails in
every reading. Dropping the pairs likely on one wire from SSI changes none of this. The SSI rests on
the assumption of fig3c-ssi-frv-correlation.

## Claims not tested here

- fig3b-firing-muhgp-higher-aversive: needs the continuous microwire signals; the dandiset holds only
  spike times and mean waveforms of the microwire data (as the data descriptor says).
- fig2-participant4-bold-correlations, table2-bold-correlations-group, table2-bold-correlation-values,
  bold-hgp-uncorrelated-low-mi, fig4c-regression-firing-bold, fig4d-hgp-bold-covariation: need the
  BOLD time courses of the clinical fMRI, which the paper names no public deposit for; the dandiset
  holds no fMRI.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig1b-delta-decrease | blocked (methods_underspecified): the z-scoring readings disagree |
| fig1c-hgp-increase-first-2s | not reproduced |
| other-bands-no-effect | not reproduced (assumption) |
| table2-mi-values | not reproduced |
| fig1d-delta-hgp-mi-correlation | not reproduced (assumption) |
| nsoz-delta-hgp-mi-inverse | not reproduced (assumption) |
| table2-delta-reduced-12-of-14 | partially reproduced (assumption) |
| table2-hgp-mi-positive-nsoz | not reproduced (assumption) |
| fig1e-hgp-not-delta-distinguishes-soz | not reproduced (assumption) |
| hgp-soz-sensitivity-specificity | not reproduced (assumption) |
| fig1f-hgp-more-specific-than-delta | blocked (methods_underspecified): the readings disagree |
| neuron-count | partially reproduced |
| fig3a-firing-increase-aversive | not reproduced |
| fig3a-firing-gradual-sustained | not reproduced |
| fig3b-firing-hgp-no-condition-difference | partially reproduced |
| fig3c-ssi-frv-correlation | not reproduced (assumption) |
| fig3d-firing-correlations-modalities | partially reproduced (HGP part only) |
| fig4b-regression-firing-hgp | not reproduced (assumption) |
| fig3b-firing-muhgp-higher-aversive | blocked (data_missing) |
| fig2-participant4-bold-correlations, table2-bold-correlations-group, table2-bold-correlation-values, bold-hgp-uncorrelated-low-mi, fig4c-regression-firing-bold, fig4d-hgp-bold-covariation | blocked (data_missing) |

How the analysis differs from the paper's. The paper links no code. Where its Methods leave a step
open, every reading named in the claims file was run, and the outcomes are judged on all of them;
fixed MI windows taken from the Results' text were run as a further reading, and two z-scoring
variants (`trialsd`, `baselinepool`) and the two other baselines of Methods 4.8 are shown without
deciding any outcome. The z-scoring decides most iEEG outcomes: z-scoring over the whole trial
removes the sustained delta decrease, while z-scoring against the baseline keeps it. The paper's MI
window, the group clusters, does not exist for HGP here, so the claims on the HGP MI rest on the fixed
windows, an assumption. The group test is a within-patient form of the paper's mixed model, which gives the same condition
estimate for this balanced design (t differs slightly). SSI uses every other unit of the amygdala, as the dandiset does not say which
microwire a unit was on. Epochs in which a unit has no spike at all are taken as not recorded.

What was found about the dandiset. Each session is deposited twice with identical data. The iEEG is
stated in volts but holds microvolts. The 26 s epochs are laid end to end with jumps at their
boundaries, and the trial table puts the first trial's start at its baseline and the others' at
block onset. The units' observation intervals cover all 17 epochs although the 16 units of subject 7
have no spikes after 286 s and four units of subject 2 none in the first epoch. Two ages differ from
the paper (subject 2's file gives 39, the age of the paper's patient 7), five channel locations name
the hippocampus or are unknown for channels the paper places in the amygdala, no unit is assigned to
a microwire, and the units are in 8 amygdalae of 5 patients, not the 9 amygdalae of the Results nor
the amygdalae of Table 1 (subject 7 has units, 5 and 8 none). Re-mapping subjects 2 and 7, or 7 to the
paper's 5 and 8, does not bring the MIs closer to Table 2. The BOLD time courses and the continuous
microwire signals are not in the dandiset.

In [ ]:
print(f"Notebook run time {(time.time() - started) / 60:.1f} min")